# Data Quality Tests: People Data Lake

This notebook runs the same checks as `tests/test_checks.py`, step by step, so you can see the data at each layer and the results of each check.

The data is synthetic. It has intentional defects, described in `data/README.md`.

In [ ]:
import sys
from pathlib import Path

import duckdb
import pandas as pd

# Run from the notebooks/ folder or from the project root
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

from qa.checks import CHECKS, run_checks
from src.pipeline import build_layers

print('Project root:', ROOT)

## 1. Build the layers

Load the raw extract into Bronze, Silver, and Gold, then check the row counts.

In [ ]:
con = duckdb.connect()
build_layers(con)

for table in ['bronze_workday', 'silver_worker', 'gold_worker_current']:
    n = con.sql(f'SELECT COUNT(*) FROM {table}').fetchall()[0][0]
    print(f'{table}: {n} rows')

con.sql('SELECT * FROM silver_worker ORDER BY worker_id, effective_date').df()

## 2. Run all checks

Each check returns PASS or FAIL, the number of violations, and sample rows.

In [ ]:
results = run_checks(con)
results_df = pd.DataFrame(results)
results_df[['check', 'status', 'violations']]

In [ ]:
# Show the sample rows for each failed check
failed = results_df[results_df['status'] == 'FAIL']
failed[['check', 'sample']]

## 3. Tests

Each test below matches a test in `tests/test_checks.py`. Run the cell to check the expected results.

In [ ]:
RESULTS = {r['check']: r for r in results}

EXPECTED_FAIL = [
    'duplicate_worker_effective_date',
    'overlapping_effective_periods',
    'end_before_start',
    'missing_hire_date',
    'missing_job_code',
    'orphan_manager',
    'sin_not_masked',
]

EXPECTED_PASS = [
    'reconciliation_row_count',
    'gold_one_row_per_worker',
]


def test_known_defects_are_detected():
    for name in EXPECTED_FAIL:
        assert RESULTS[name]['status'] == 'FAIL', name
        assert RESULTS[name]['violations'] >= 1, name
    print('PASS: test_known_defects_are_detected')


def test_clean_checks_pass():
    for name in EXPECTED_PASS:
        assert RESULTS[name]['status'] == 'PASS', name
    print('PASS: test_clean_checks_pass')


def test_every_check_is_covered():
    assert {c.name for c in CHECKS} == set(EXPECTED_FAIL) | set(EXPECTED_PASS)
    assert set(RESULTS) == {c.name for c in CHECKS}
    print('PASS: test_every_check_is_covered')


def test_clean_worker_is_not_flagged():
    rows = con.sql("SELECT worker_id FROM silver_worker WHERE worker_id = 'W1002'").fetchall()
    assert len(rows) == 2
    assert 'W1002' not in RESULTS['overlapping_effective_periods']['sample']
    assert 'W1002' not in RESULTS['duplicate_worker_effective_date']['sample']
    print('PASS: test_clean_worker_is_not_flagged')

In [ ]:
tests = [
    test_known_defects_are_detected,
    test_clean_checks_pass,
    test_every_check_is_covered,
    test_clean_worker_is_not_flagged,
]

for test in tests:
    test()

print(f'\n{len(tests)} test functions passed')